In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from sentence_transformers import SentenceTransformer, util
EMBEDDING_MODEL = "ai-forever/FRIDA"

embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")

In [3]:
!pip install -q rouge nltk pymorphy2

In [4]:
from typing import List
from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import json

file_path = "/content/drive/MyDrive/data/results.json"
# Load JSON data
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

**BLEU and ROUGH**

In [5]:
# Initialize metrics
rouge = Rouge()
bleu_scores = []
rouge_scores = []

# Calculate metrics
for entry in data:
    reference = entry["ground_true_answer"]
    hypothesis = entry["answer"]

    # Skip if reference is empty
    if not reference:
        continue

    # BLEU score
    smoothie = SmoothingFunction().method4
    bleu = sentence_bleu([reference.split()], hypothesis.split(), smoothing_function=smoothie)
    bleu_scores.append(bleu)

    # ROUGE score
    rouge_score = rouge.get_scores(hypothesis, reference)[0]
    rouge_scores.append(rouge_score)

# Average scores
average_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
average_rouge = {
    "rouge-1": {
        "f": sum(score["rouge-1"]["f"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "p": sum(score["rouge-1"]["p"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "r": sum(score["rouge-1"]["r"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
    },
    "rouge-2": {
        "f": sum(score["rouge-2"]["f"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "p": sum(score["rouge-2"]["p"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "r": sum(score["rouge-2"]["r"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
    },
    "rouge-l": {
        "f": sum(score["rouge-l"]["f"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "p": sum(score["rouge-l"]["p"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "r": sum(score["rouge-l"]["r"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
    },
}

average_bleu, average_rouge

(0.047828310921452996,
 {'rouge-1': {'f': 0.21697527665867533,
   'p': 0.16321211661267634,
   'r': 0.41941298526128123},
  'rouge-2': {'f': 0.09817091249008952,
   'p': 0.07264288349315295,
   'r': 0.2087163105321249},
  'rouge-l': {'f': 0.20534235298537323,
   'p': 0.15424827694595114,
   'r': 0.39961971381684236}})

**AVG SIMILARITY**

In [7]:
# Подсчёт семантического сходства между ответами модели и правильными ответами
similarities = []
for entry in data:
    reference = entry["ground_true_answer"]
    hypothesis = entry["answer"]
    emb_ref = embedding_model.encode(reference, convert_to_tensor=True)
    emb_hyp = embedding_model.encode(hypothesis, convert_to_tensor=True)
    similarity = float(util.cos_sim(emb_ref, emb_hyp))
    similarities.append(similarity)

# Среднее значение семантического сходства
average_similarity = sum(similarities) / len(similarities)
average_similarity


0.5462037638073491

**Context precision**

In [8]:
relevant = 0
total = 0

for entry in data:
    gt = entry["ground_true_answer"]
    gt_emb = embedding_model.encode(gt, convert_to_tensor=True)

    for doc in entry["retrieved_documents"]:
        doc_emb = embedding_model.encode(doc["text"], convert_to_tensor=True)
        sim = util.cos_sim(gt_emb, doc_emb).item()
        if sim > 0.7:
            relevant += 1
        total += 1

print(f"Context Precision (SNR, semantically): {relevant / total:.3f}")

KeyboardInterrupt: 

**Context recall**

In [ ]:
total = len(data)
relevant_found = 0

for entry in data:
    gt = entry["ground_true_answer"]
    gt_emb = embedding_model.encode(gt, convert_to_tensor=True)
    found = False
    for doc in entry["retrieved_documents"]:
        doc_emb = embedding_model.encode(doc["text"], convert_to_tensor=True)
        sim = util.cos_sim(gt_emb, doc_emb).item()
        if sim > 0.7:
            found = True
            break
    if found:
        relevant_found += 1

print(f"Context Recall (semantic): {relevant_found / total:.3f}")

**Faithfulness**

In [ ]:
faithful = 0
total = 0

for entry in data:
    ans = entry["answer"]
    ans_emb = embedding_model.encode(ans, convert_to_tensor=True)

    supported = False
    for doc in entry["retrieved_documents"]:
        doc_emb = embedding_model.encode(doc["text"], convert_to_tensor=True)
        sim = util.cos_sim(ans_emb, doc_emb).item()
        if sim > 0.7:
            supported = True
            break

    if supported:
        faithful += 1
    total += 1

print(f"Faithfulness (semantic): {faithful / total:.3f}")

**Answer Relevance**

In [ ]:
relevance_scores = []

for entry in data:
    question = entry["question"]
    answer = entry["answer"]

    emb_q = embedding_model.encode(question, convert_to_tensor=True)
    emb_a = embedding_model.encode(answer, convert_to_tensor=True)

    similarity = util.cos_sim(emb_q, emb_a).item()
    relevance_scores.append(similarity)

average_relevance = sum(relevance_scores) / len(relevance_scores)
print(f"Average Answer Relevance: {average_relevance:.3f}")

**Answer similarity**

In [ ]:
similarities = []

for entry in data:
    answer = entry["answer"]
    ground_truth = entry["ground_true_answer"]

    emb1 = embedding_model.encode(answer, convert_to_tensor=True)
    emb2 = embedding_model.encode(ground_truth, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()
    similarities.append(similarity)

average_similarity = sum(similarities) / len(similarities)
print(f"Answer Similarity (semantic): {average_similarity:.3f}")

**Answer correctness**

In [ ]:
correct_answers = 0

for entry in data:
    answer = entry["answer"]
    ground_truth = entry["ground_true_answer"]

    emb1 = embedding_model.encode(answer, convert_to_tensor=True)
    emb2 = embedding_model.encode(ground_truth, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()
    if similarity > 0.8:
        correct_answers += 1

correctness_score = correct_answers / len(data)
print(f"Answer Correctness (semantic threshold): {correctness_score:.3f}")